In [1]:
from openai import OpenAI

client = OpenAI()

In [7]:
from pathlib import Path

ontology_text = Path(
    "../resouces/firmable_ontology.yaml"
).read_text(encoding="utf-8")

In [8]:
prompt = f"""
You are a schema mapping expert.

Map the source dataset to the Firmable canonical ontology.

Rules:
- Map only fields supported by evidence.
- A target field may use one or multiple source fields.
- Do not guess uncertain mappings.
- Source-specific transformation happens before normalization.
- General normalization is handled later by code.
- Prefer DuckDB SQL expressions for source-specific transformations.
- Keep unmapped source fields and explain why.
- Give mapping confidence from 0 to 1.

ONTOLOGY:
{ontology_text}

DATASET PROFILE:
{json.dumps(profile, indent=2)}
"""

In [9]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-6-luna",
    input=prompt
)

result = response.output_text

print(result)

## Recommended mappings

Treat each ASIC row as a source observation. Do not collapse rows sharing an ACN or reconcile historical and current names during ingestion.

| Canonical field | Source field(s) | Source-specific transformation | Confidence |
|---|---|---|---:|
| `entity.legal_name` | `Company Name` | Trim surrounding whitespace; emit the value as the name stated for that row. Do not assume it is current when `Current Name Indicator` is not `Y`. | 0.98 |
| `entity.legal_name` | `Current Name` | Emit a **separate** legal-name observation when nonblank and not the literal null sentinel `nan`. This field explicitly supplies a current name on rows where it is populated. | 0.95 |
| `entity.acn` | `ACN` | Trim whitespace; retain as text to preserve leading zeros. Require nine digits; perform ACN checksum validation in the validation/normalization stage. | 0.99 |
| `entity.abn` | `ABN` | Cast the integer to text. Treat `0` as missing; emit only an 11-digit value. Validate the ABN chec

In [10]:
result

'## Recommended mappings\n\nTreat each ASIC row as a source observation. Do not collapse rows sharing an ACN or reconcile historical and current names during ingestion.\n\n| Canonical field | Source field(s) | Source-specific transformation | Confidence |\n|---|---|---|---:|\n| `entity.legal_name` | `Company Name` | Trim surrounding whitespace; emit the value as the name stated for that row. Do not assume it is current when `Current Name Indicator` is not `Y`. | 0.98 |\n| `entity.legal_name` | `Current Name` | Emit a **separate** legal-name observation when nonblank and not the literal null sentinel `nan`. This field explicitly supplies a current name on rows where it is populated. | 0.95 |\n| `entity.acn` | `ACN` | Trim whitespace; retain as text to preserve leading zeros. Require nine digits; perform ACN checksum validation in the validation/normalization stage. | 0.99 |\n| `entity.abn` | `ABN` | Cast the integer to text. Treat `0` as missing; emit only an 11-digit value. Validate th